In [3]:
# !pip install langchain-google-genai

In [1]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate



In [2]:
# Test ollama 
# from langchain_ollama import ChatOllama

# llm = ChatOllama(
#     model="qwen2.5:1.5b",
#     temperature=0
# )

# response = llm.invoke("Explain what an AWS SQS queue does")
# print(response.content)

# Practice: Simple Text Analyzer

Create a sequential graph with 3 nodes:
```
User Text
   ↓
Node 1: Summarize
   ↓
Node 2: Extract Keywords
   ↓
Node 3: Generate Title
   ↓
Final Result
```

In [3]:
# initialize the LLM with the model and temperature
llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0
)

In [4]:
class TextSummarizationGraphState(BaseModel):
    text: str
    summary: str = None
    keywords: list[str] = None
    title: str = None

In [5]:
# Define a function to summarize text using the LLM

def summarize_text(state: TextSummarizationGraphState) -> TextSummarizationGraphState:
    prompt_template = PromptTemplate(
        input_variables=["text"],
        template="Summarize the following text: {text}"
    )

    text = state.text

    chain = prompt_template | llm
    
    # Use the LLM to summarize the text
    summary = chain.invoke({"text": text})
    
    # Update the state with the summary
    state.summary = summary.content
    return state


def extract_keywords(state: TextSummarizationGraphState) -> TextSummarizationGraphState:
    prompt_template = PromptTemplate(
        input_variables=["summary"],
        template="Extract keywords from the following text and give me comma-separated values: {summary}"
    )

    summary = state.summary
    
    chain = prompt_template | llm

    # Use the LLM to extract keywords
    keywords = chain.invoke({"summary": summary})
    state.keywords = keywords.content.split(", ")

    return state


def generate_title(state: TextSummarizationGraphState) -> TextSummarizationGraphState:
    prompt_template = PromptTemplate(
        input_variables=["summary", "keywordList"],
        template="""Generate a title for the following summary text and keywords 
        
            text: {summary}

            keywords: {keywordList}
            
            """
    )

    summary = state.summary
    keywordList = state.keywords
    
    chain = prompt_template | llm

    # Use the LLM to generate a title
    title = chain.invoke({"summary": summary, "keywordList": keywordList})
    state.title = title.content

    return state

In [6]:
# initialize graph
graph = StateGraph(TextSummarizationGraphState)

# add nodes to the graph
graph.add_node("summarize_text", summarize_text)
graph.add_node("extract_keywords", extract_keywords)
graph.add_node("generate_title", generate_title)

# define the edges of the graph
graph.add_edge(START, "summarize_text")
graph.add_edge("summarize_text", "extract_keywords")
graph.add_edge("extract_keywords", "generate_title")
graph.add_edge("generate_title", END)

# show the graph flow
# graph.compile()

In [7]:
# compile graph

final_graph = graph.compile()

In [8]:
# initial state

initialState = TextSummarizationGraphState(text="""Amazon S3 is an object storage service. It provides high durability
and can store virtually unlimited amounts of data.""")

In [9]:
output = final_graph.invoke(initialState)



In [10]:
for key in output.keys():
    print(f"{key}: {output[key]}\n")

text: Amazon S3 is an object storage service. It provides high durability
and can store virtually unlimited amounts of data.

summary: Amazon S3 is an object storage service that offers high durability and can store an unlimited amount of data.

keywords: ['Amazon S3', 'object', 'storage', 'service', 'durability', 'data', 'unlimited', 'data.']

title: Title: "Amazon S3: A Comprehensive Overview of Object Storage Service with High Durability and Unlimited Data Capacity"

